[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shuishen112/neural-networks/blob/main/exercises/05_regularization_exercises.ipynb)

# Exercises 5 — Regularisation

Fill in every cell marked `# YOUR CODE HERE` and answer the written questions.
The `assert` cells check your work: if a cell runs without an error, you are
probably on the right track.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

## Exercise 5.1 — Inverted dropout

Implement `MyDropout`, an `nn.Module` with the behaviour of `nn.Dropout(p)`:

* in **training** mode (`self.training` is `True`), zero every element
  independently with probability `p` and scale the survivors by $1/(1-p)$;
* in **evaluation** mode, return the input unchanged.

Handle the edge case `p = 0` (identity). *Hint:* `torch.rand_like(x) >= p` gives a
boolean mask whose entries are `True` with probability $1-p$.

In [ ]:
class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        assert 0.0 <= p < 1.0
        self.p = p

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
d = MyDropout(0.3)
x = torch.ones(2000, 500)
d.train()
out = d(x)
frac_zero = (out == 0).float().mean().item()
assert abs(frac_zero - 0.3) < 0.01, frac_zero
assert abs(out.mean().item() - 1.0) < 0.01, out.mean().item()              # expectation preserved
assert torch.allclose(out[out != 0], torch.full_like(out[out != 0], 1 / 0.7))
d.eval()
assert torch.equal(d(x), x)
assert torch.equal(MyDropout(0.0).train()(x), x)
# gradients flow only through the kept units, with the same scaling
xg = torch.ones(10, 10, requires_grad=True)
d.train(); y = d(xg); y.sum().backward()
assert torch.allclose(xg.grad, (y != 0).float() / 0.7)
print("MyDropout OK")

**Question.** Each unit of a layer with $n$ inputs $h_j$ is followed by a linear
layer computing $a = \sum_j w_j \tilde h_j$. With dropout (rate $p$, inverted
scaling), what are $\mathbb E[a]$ and $\operatorname{Var}[a]$ over the random mask,
for fixed $h$ and $w$? What does this say about the "noise" dropout injects?

*Your answer here.*

## Exercise 5.2 — Batch normalisation from scratch

Implement `batchnorm_train(x, gamma, beta, running_mean, running_var, momentum, eps)`
for an input of shape `(B, D)` in **training mode**. It must

1. normalise every feature with the **biased** batch variance,
2. apply the affine transform $\gamma\hat x + \beta$,
3. update the running statistics **in place** like PyTorch does:
   `running = (1 - momentum) * running + momentum * batch_stat`, where the running
   variance is updated with the **unbiased** batch variance ($\frac{B}{B-1}\sigma^2$),

and return the output. Then implement `batchnorm_eval`, which uses the running
statistics. We compare against `nn.BatchNorm1d`, including the gradients.

In [ ]:
def batchnorm_train(x, gamma, beta, running_mean, running_var, momentum=0.1, eps=1e-5):
    # YOUR CODE HERE
    raise NotImplementedError()


def batchnorm_eval(x, gamma, beta, running_mean, running_var, eps=1e-5):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
B, D = 32, 5
ref = nn.BatchNorm1d(D)
with torch.no_grad():
    ref.weight.copy_(torch.rand(D) + 0.5)
    ref.bias.copy_(torch.randn(D))
gamma = ref.weight.detach().clone().requires_grad_()
beta = ref.bias.detach().clone().requires_grad_()
rm, rv = torch.zeros(D), torch.ones(D)

ref.train()
for step in range(5):
    x = (3 * torch.randn(B, D) + 2).requires_grad_()
    x2 = x.detach().clone().requires_grad_()
    y_ref = ref(x)
    y_my = batchnorm_train(x2, gamma, beta, rm, rv)
    assert torch.allclose(y_ref, y_my, atol=1e-5)
    w = torch.randn(B, D)
    (y_ref * w).sum().backward()
    (y_my * w).sum().backward()
    assert torch.allclose(x.grad, x2.grad, atol=1e-5)
assert torch.allclose(rm, ref.running_mean, atol=1e-5)
assert torch.allclose(rv, ref.running_var, atol=1e-5)
assert torch.allclose(gamma.grad, ref.weight.grad, atol=1e-4)
assert torch.allclose(beta.grad, ref.bias.grad, atol=1e-4)

ref.eval()
xt = torch.randn(3, D)
assert torch.allclose(ref(xt), batchnorm_eval(xt, gamma, beta, rm, rv), atol=1e-5)
print("batch norm OK")

**Question.** In training mode, what does batch norm output for a batch of size 1?
Try `nn.BatchNorm1d(3).train()(torch.randn(1, 3))` and explain the result.

*Your answer here.*

## Exercise 5.3 — An early-stopping helper

Implement the class `EarlyStopping`. Calling `stopper.step(val_loss, model)` after
every epoch should

* if `val_loss < best - min_delta`: record the new best loss, store a **deep copy**
  of `model.state_dict()` in `self.best_state`, reset the counter, and return `False`;
* otherwise increase the counter and return `True` once it reaches `patience`
  (meaning "stop now"), `False` before that.

`stopper.restore(model)` loads the best state back into the model.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best = float("inf")
        self.best_state = None
        self.counter = 0

    def step(self, val_loss, model):
        # YOUR CODE HERE
        raise NotImplementedError()

    def restore(self, model):
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
lin = nn.Linear(2, 1)
es = EarlyStopping(patience=3)
decisions = []
for i, loss in enumerate([1.0, 0.8, 0.9, 0.7, 0.75, 0.71, 0.72, 0.6]):
    with torch.no_grad():
        lin.bias.fill_(float(i))               # mark the model with the epoch number
    decisions.append(es.step(loss, lin))
assert decisions == [False, False, False, False, False, False, True, False], decisions
assert es.best == 0.6
es2 = EarlyStopping(patience=3)
for i, loss in enumerate([1.0, 0.5, 0.6, 0.7, 0.8]):
    with torch.no_grad():
        lin.bias.fill_(float(i))
    if es2.step(loss, lin):
        break
assert i == 4
es2.restore(lin)
assert lin.bias.item() == 1.0, "restore() must load the state of the best epoch (a deep copy!)"
print("EarlyStopping OK")

## Exercise 5.4 — A regularisation experiment

We use the setting of the lecture: 300 training digits, 20 % of them with random
labels, and a clean validation set. The functions `make_model` and `run` are given.

Train **three** configurations for 100 epochs each, using your `EarlyStopping`
(patience 15) to restore the best model at the end:

* `"baseline"`: `make_model(dropout=0.0)` with `Adam(lr=1e-3)`;
* `"dropout"`: `make_model(dropout=0.5)` with `Adam(lr=1e-3)`;
* `"dropout+wd"`: `make_model(dropout=0.5)` with `AdamW(lr=1e-3, weight_decay=1.0)`.

Store the validation accuracy of each restored model in the dictionary `results`,
and the model itself in `trained`. Use `torch.manual_seed(0)` before creating each model.

In [ ]:
digits = load_digits()
X_all = (digits.data / 16.0).astype(np.float32)
y_all = digits.target.astype(np.int64)
X_pool, X_test, y_pool, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=0, stratify=y_all)
X_rest, X_val, y_rest, y_val = train_test_split(X_pool, y_pool, test_size=0.3, random_state=0, stratify=y_pool)
X_small, y_small = X_rest[:300], y_rest[:300].copy()
flip = rng.random(300) < 0.2
y_small[flip] = rng.integers(0, 10, flip.sum())

train_ds = TensorDataset(torch.from_numpy(X_small), torch.from_numpy(y_small))
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)), batch_size=512)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)), batch_size=512)


def make_model(dropout=0.0):
    return nn.Sequential(
        nn.Linear(64, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 10),
    ).to(device)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total += F.cross_entropy(logits, yb, reduction="sum").item()
        correct += (logits.argmax(1) == yb).sum().item()
        n += len(yb)
    return total / n, correct / n


def run(model, optimizer, stopper, n_epochs=100):
    """Train with early stopping; returns the list of validation losses."""
    loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(0))
    val_losses = []
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        val_loss, _ = evaluate(model, val_loader)
        val_losses.append(val_loss)
        if stopper.step(val_loss, model):
            break
    stopper.restore(model)
    return val_losses

In [ ]:
results, trained, curves = {}, {}, {}
# YOUR CODE HERE
raise NotImplementedError()

for name, c in curves.items():
    plt.plot(c, label=name)
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.show()
for name, acc in results.items():
    print(f"{name:12s} val acc {acc:.3f} | test acc {evaluate(trained[name], test_loader)[1]:.3f}")

In [ ]:
assert set(results) == {"baseline", "dropout", "dropout+wd"}
assert all(0.0 <= a <= 1.0 for a in results.values())
assert all(not m.training for m in trained.values())
best_name = max(results, key=results.get)
best_test = evaluate(trained[best_name], test_loader)[1]
print("best configuration:", best_name, "| test accuracy", round(best_test, 3))
assert best_test >= 0.78
print("experiment OK")

**Question.** We selected the best configuration by its *validation* accuracy and
reported its *test* accuracy. Why would reporting the best of the three *test*
accuracies be optimistic? Also, early stopping itself used the validation set —
is the validation accuracy of the restored model an unbiased estimate?

*Your answer here.*

## Exercise 5.5 — Written questions

**(a)** Show that for plain SGD (no momentum), adding $\frac{\lambda}{2}\lVert\mathbf
w\rVert^2$ to the loss is identical to multiplying the weights by $(1-\eta\lambda)$
before each gradient step. Why is this no longer true for Adam?

*Your answer here.*

**(b)** A colleague evaluates a network containing batch norm and dropout but
forgets `model.eval()`. List two distinct ways in which the reported results are wrong.

*Your answer here.*

**(c)** Which of these augmentations are appropriate for (i) handwritten digit
recognition and (ii) classifying photos of animals: horizontal flip, rotation by
±10°, rotation by 180°, small translations, colour jitter? Justify briefly.

*Your answer here.*

**(d)** In the double-descent experiment of the lecture, the test error peaks when
the number of random features $p$ equals the number of training examples $n$.
Explain why, in terms of the singular values of the feature matrix.

*Your answer here.*